__Data Leakage Checker__

Something we didn't account for initially was the cross referencing of cards in the other card rulings. 
What this meant. Is that unknowingly we hade a data leakage.
As such it was important this was addressed.
We needed to ascertain what the size of the leakage was.
What cards were 'leaking' i.e. being referenced by other cards.
And then we needed to see which cards leaked onto. Thus the nature of it being cross contamination.
 


In [ ]:
import pandas as pd
import re

# Load the data and handle empty values
df = pd.read_csv("merged_mtg_rules.csv")
df['rules'] = df['rules'].fillna("")
df['name'] = df['name'].fillna("")

# Extract a master list of all valid, unique names in the dataset
all_names = [name for name in df['name'].unique().tolist() if name.strip()]

# Define the cross-contamination checker
def find_leaked_names(row):
    text = row['rules']
    target = row['name']
    
    if not text:
        return []
    
    leaks = []
    for name in all_names:
        # Fast pass: Only run regex if the name is found anywhere in the text
        if name != target and name in text:
            # Slow pass: Check word boundaries to ensure we aren't matching 
            # a substring (e.g. stopping "Void" from matching "Avoid")
            if re.search(r'\b' + re.escape(name) + r'\b', text):
                leaks.append(name)
                
    return leaks

# Apply the function to create a list of leaks and a boolean flag
df['leaked_names'] = df.apply(find_leaked_names, axis=1)
df['has_leakage'] = df['leaked_names'].apply(lambda x: len(x) > 0)

# Output the cleaned dataset
df.to_csv("mtg_rules_leakage_analysis.csv", index=False)

In [2]:

import ast

# Load the exported analysis file
df = pd.read_csv("mtg_rules_leakage_analysis.csv")

# Converts the string representation of the lists back into actual Python lists 
# so we can interact with the leaked names if we want to inspect them.
df['leaked_names'] = df['leaked_names'].apply(ast.literal_eval)

# Calculate the cross-contamination percentage
# .mean() on a boolean column gets the ratio of True values (e.g., 0.1757)
contamination_percentage = df['has_leakage'].mean() * 100

# Print the summary
print(f"Total Rows Evaluated: {len(df)}")
print(f"Rows with Cross-Contamination: {df['has_leakage'].sum()}")
print(f"Cross-Contamination Rate: {contamination_percentage:.2f}%")

# View a random sample of the contaminated rows to verify
contaminated_df = df[df['has_leakage'] == True]
print("\nSample of contaminated rows:")
print(contaminated_df[['name', 'leaked_names']].head(10))

Total Rows Evaluated: 14248
Rows with Cross-Contamination: 2504
Cross-Contamination Rate: 17.57%

Sample of contaminated rows:
                    name                            leaked_names
6          Angel's Grace             [Chalice of the Void, Void]
8        Mists of Lórien                                     [X]
12  Battle for Bretagard                                     [X]
13     Inspiring Vantage            [Warp World, Oblivion Sower]
15     Yorion, Sky Nomad                     [Phyrexian Revoker]
17          Rules Lawyer  [Rasputin Dreamweaver, Mirror Gallery]
28            Mirrorform                                     [X]
32                 Guile                [Lignify, Yixlid Jailer]
33     Decree of Justice                            [X, Justice]
35   Leyline of the Void                      [Void, Hollow One]


In [6]:
df = pd.read_csv("mtg_rules_leakage_analysis.csv")

# Convert the 'leaked_names' column from text strings back into actual Python lists
# Use lambda function to safely handle empty lists or null values
df['leaked_names'] = df['leaked_names'].apply(
    lambda x: ast.literal_eval(x) if pd.notnull(x) and x.startswith('[') else []
)

# Flatten the column of lists into one single master list containing every leaked mention
all_leaks = [name for sublist in df['leaked_names'] for name in sublist]

# Count the frequency of each name in the master list using pandas
leak_frequency = pd.Series(all_leaks).value_counts()

# Get the top 100 biggest leakers
top_100_leakers = leak_frequency.head(100).reset_index()
top_100_leakers.columns = ['Card Name', 'Number of Rows Contaminated']

print("--- Top 100 Biggest 'Leakers' ---")
print(top_100_leakers.to_string(index=False))
len(leak_frequency)

--- Top 100 Biggest 'Leakers' ---
                  Card Name  Number of Rows Contaminated
                          X                         1112
            Doubling Season                           39
                      Clone                           32
                  Sacrifice                           31
               Shapeshifter                           30
                       Void                           28
                       Fury                           22
                     Desert                           20
                      Fling                           20
                     Stifle                           17
                       Lich                           16
                  Nightmare                           16
                 Earthquake                           15
                   Sentinel                           15
        Leyline of the Void                           13
                   Betrayal                           

992

In [7]:
df = pd.read_csv("mtg_rules_leakage_analysis.csv")

# Safely parse the leaked_names lists
df['leaked_names'] = df['leaked_names'].apply(
    lambda x: ast.literal_eval(x) if pd.notnull(x) and x.startswith('[') else []
)

# Flatten the list to find the top 100 leakers globally
all_leaks = [name for sublist in df['leaked_names'] for name in sublist]
# Create a fast-lookup Set of the top 100 most frequent names
top_100_leakers = set(pd.Series(all_leaks).value_counts().head(100).index)

# Define a check to see if a row's leaks intersect with the top 100
def has_top_100_leak(leak_list):
    # Returns True if ANY name in this row's leak_list is in the top_100_leakers set
    return any(name in top_100_leakers for name in leak_list)

# Apply the flag to the DataFrame
df['has_top_100_leak'] = df['leaked_names'].apply(has_top_100_leak)

# Filter out the rows that got flagged
top100_clean_df = df[~df['has_top_100_leak']].copy()

# Save the new dataset
top100_clean_df.to_csv("mtg_rules_top100_clean.csv", index=False)

print(f"Original row count: {len(df)}")
print(f"Top 100 clean row count: {len(top100_clean_df)}")
print(f"Removed {len(df) - len(top100_clean_df)} rows contaminated by the top 100 leakers.")

Original row count: 14248
Top 100 clean row count: 12512
Removed 1736 rows contaminated by the top 100 leakers.


,uuid,frameVersion,setCode,isOnlineOnly,isTextless,keywords,name,text,type,rules,leaked_names,has_leakage,has_top_100_leak
0,18d320b3-3b97-5cfc-959c-66e22911627b,2015,RIX,False,False,NaN,Reckless Rage,Reckless Rage deals 4 damage to target creatur...,Instant,You can’t cast Reckless Rage unless you choose...,[],False,False
1,e4eb8663-1d62-5abc-bbfe-ff9dd2de43c0,2015,PTHB,False,False,"Flying, Lifelink",Dream Trawler,"Flying, lifelink\nWhenever you draw a card, th...",Creature — Sphinx,You can activate Dream Trawler's last ability ...,[],False,False
2,64f3de78-c603-5aa9-bd74-9feafc32f476,2015,NCC,False,False,NaN,Generous Gift,Destroy target permanent. Its controller creat...,Instant,If the target permanent is an illegal target b...,[],False,False
3,38d1f975-1b3b-5047-8328-61f5ca71ddba,2015,SLD,False,False,NaN,Leeching Sliver,"Whenever a Sliver you control attacks, defendi...",Creature — Sliver,"Abilities that Slivers grant, as well as power...",[],False,False
4,9729bc88-7ca2-5ad1-bcf4-fd5c2b88f0ad,2015,SNC,False,False,"Haste, Deathtouch",Maestros Diabolist,"Deathtouch, haste\nWhenever this creature atta...",Creature — Vampire Warrior,Maestros Diabolist's triggered ability has an ...,[],False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
14243,9bf0b366-1471-5484-8016-fe1e5e66ccaa,2015,WOE,False,False,Bargain,Johann's Stopgap,"Bargain (You may sacrifice an artifact, enchan...",Sorcery,The cost-reduction ability of Johann's Stopgap...,[],False,False
14244,233784b1-3db1-5ed6-9262-04c55217c7d9,1997,APC,False,False,NaN,Ceta Sanctuary,"At the beginning of your upkeep, if you contro...",Enchantment,"The ability has you draw one or two cards, but...",[],False,False
14245,f811931c-810f-58cd-86e4-50be261ec920,2015,KTK,False,False,NaN,Smoke Teller,{1}{U}: Look at target face-down creature.,Creature — Human Shaman,You could target a creature you control with S...,[Smoke],True,False
14246,6fb939cc-6091-5399-9106-fa863d212cb9,2003,MRD,False,False,Entwine,Journey of Discovery,Choose one —\n• Search your library for up to ...,Sorcery,If you cast an entwined Journey of Discovery u...,[Vedalken Orrery],True,False


In [11]:
# 1. Load the exported analysis file
df = pd.read_csv("mtg_rules_top100_clean.csv")

# Optional: Safely convert the string representation of the lists back into actual Python lists 
# so you can interact with the leaked names if you want to inspect them.
df['leaked_names'] = df['leaked_names'].apply(ast.literal_eval)

# 2. Calculate the cross-contamination percentage
# .mean() on a boolean column gets the ratio of True values (e.g., 0.1757)
contamination_percentage = df['has_leakage'].mean() * 100

# 3. Print the summary
print(f"Total Rows Evaluated: {len(df)}")
print(f"Rows with Cross-Contamination: {df['has_leakage'].sum()}")
print(f"Cross-Contamination Rate: {contamination_percentage:.2f}%")

# 4. View a random sample of the contaminated rows to verify
contaminated_df = df[df['has_leakage'] == True]
print("\nSample of contaminated rows:")
print(contaminated_df[['name', 'leaked_names']].head(10))

Total Rows Evaluated: 14248
Rows with Cross-Contamination: 2504
Cross-Contamination Rate: 17.57%

Sample of contaminated rows:
                    name                            leaked_names
6          Angel's Grace             [Chalice of the Void, Void]
8        Mists of Lórien                                     [X]
12  Battle for Bretagard                                     [X]
13     Inspiring Vantage            [Warp World, Oblivion Sower]
15     Yorion, Sky Nomad                     [Phyrexian Revoker]
17          Rules Lawyer  [Rasputin Dreamweaver, Mirror Gallery]
28            Mirrorform                                     [X]
32                 Guile                [Lignify, Yixlid Jailer]
33     Decree of Justice                            [X, Justice]
35   Leyline of the Void                      [Void, Hollow One]


In [10]:
df = pd.read_csv("mtg_rules_leakage_analysis.csv")

# Filter out EVERY row that has ANY leakage (keeping only where has_leakage is False)
fully_clean_df = df[~df['has_leakage']].copy()

# Save the 100% clean dataset
fully_clean_df.to_csv("mtg_rules_fully_clean.csv", index=False)

print(f"Total original rows: {len(df)}")
print(f"Final fully clean rows: {len(fully_clean_df)}")
print(f"Total contaminated rows removed across all passes: {len(df) - len(fully_clean_df)}")

Total original rows: 14248
Final fully clean rows: 11744
Total contaminated rows removed across all passes: 2504


Final contamination check

In [12]:
df = pd.read_csv("mtg_rules_fully_clean.csv")
df['leaked_names'] = df['leaked_names'].apply(ast.literal_eval)

# Calculate the cross-contamination percentage
# .mean() on a boolean column gets the ratio of True values (e.g., 0.1757)
contamination_percentage = df['has_leakage'].mean() * 100

# 3. Print the summary
print(f"Total Rows Evaluated: {len(df)}")
print(f"Rows with Cross-Contamination: {df['has_leakage'].sum()}")
print(f"Cross-Contamination Rate: {contamination_percentage:.2f}%")

# View a random sample of the contaminated rows to verify
contaminated_df = df[df['has_leakage'] == True]
print("\nSample of contaminated rows:")
print(contaminated_df[['name', 'leaked_names']].head(10))

Total Rows Evaluated: 11744
Rows with Cross-Contamination: 0
Cross-Contamination Rate: 0.00%

Sample of contaminated rows:
Empty DataFrame
Columns: [name, leaked_names]
Index: []
